In [1]:
import torch
import numpy as np

import torch.nn as nn
from torch.utils.data import DataLoader
from torch.optim import AdamW

from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup, pipeline

from datasets import load_dataset
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

c:\Users\dhanr\OneDrive\Documents\ML\FinTech-model\fin_env\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# DEVICE CONFIGURATION

In [2]:
device = torch.device("cude" if torch.cuda.is_available() else "cpu")

In [3]:
print(device)

cpu


# LOAD DATASET

In [4]:
dataset = load_dataset(
    "financial_phrasebank",
    "sentences_allagree"
)

print(dataset)

c:\Users\dhanr\OneDrive\Documents\ML\FinTech-model\fin_env\lib\site-packages\datasets\load.py:1486: FutureWarning: The repository for financial_phrasebank contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/financial_phrasebank
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


DatasetDict({
    train: Dataset({
        features: ['sentence', 'label'],
        num_rows: 2264
    })
})


# TRAIN / VALIDATION SPLIT

In [5]:
dataset = dataset["train"].train_test_split(
    test_size = 0.2,
    seed = 42
)

train_dataset = dataset["train"]
val_dataset = dataset["test"]

print(len(train_dataset))
print(len(val_dataset))

1811
453


# LOAD FINBERT

In [6]:
model_name = "ProsusAI/finbert"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels = 3)

model.to(device)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14350.12it/s]


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

# TOKENIZATION

In [7]:
def tokenize(batch):
    return tokenizer(
        batch["sentence"],
        padding = "max_length",
        truncation = True,
        max_length = 128
    )


train_dataset = train_dataset.map(tokenize, batched = True)
val_dataset = val_dataset.map(tokenize, batched = True)

#renaming the columns from label to labels

train_dataset = train_dataset.rename_column("label", "labels")
val_dataset = val_dataset.rename_column("label", "labels")



In [8]:
# setting torch format

train_dataset.set_format(
    type = "torch",
    columns = ["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type = "torch",
    columns = ["input_ids", "attention_mask", "labels"]
)


# DATALOADERS

In [9]:
train_loader = DataLoader(train_dataset, batch_size = 16, shuffle = True)
val_loader = DataLoader(val_dataset, batch_size = 16) 

# OPTIMIZER + SCHEDULER

In [10]:
optimizer = AdamW(model.parameters(), lr = 2e-5)
epochs = 5
total_steps = len(train_loader) * epochs

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps = 0,
    num_training_steps = total_steps
)

# TRIANING LOOP

In [12]:
best_accuracy = 0

for epoch in range(epochs):
    print(f"\n========== Epoch {epoch+1}/{epochs} ==========")
    model.train()
    total_train_loss = 0
    train_progress = tqdm(train_loader)
    
    for batch in train_progress:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(
            input_ids = input_ids,
            attention_mask = attention_mask,
            labels = labels
        )

        loss = outputs.loss
        total_train_loss += loss.item()
        loss.backward()

        nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()

        train_progress.set_postfix({"loss": loss.item()})

    avg_train_loss = total_train_loss / len(train_loader)
    print(f"\nAverage Train Loss: {avg_train_loss:.4f}")
    



========== Epoch 1/5 ==========


  1%|          | 1/114 [00:26<50:30, 26.82s/it, loss=3.56]


KeyboardInterrupt: 